**Ciclo de Vida de los Datos con Pandas**

# Bloque 1: Configuración y Creación de Datos (Buenas Prácticas)
Temas: 1, 8, 9

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# 2. Creación de un dataset de ejemplo (Simulación de tienda)
rng = np.random.default_rng(42)
n = 100
data = {
    'id_transaccion': range(1, n+1),
    'fecha': pd.date_range(start='2026-01-01', periods=n, freq='D'),
    'cliente': [f'  Usuario_{i}  ' for i in rng.integers(1, 20, n)], # Texto sucio
    'monto': rng.uniform(10, 500, n),
    'categoria': rng.choice(['Electrónica', 'Hogar', 'Moda', np.nan], n, p=[0.3, 0.3, 0.3, 0.1])
}

df = pd.DataFrame(data)

# Introducir nulos aleatorios en 'monto'
df.loc[rng.choice(n, 10), 'monto'] = np.nan

print("Dataset cargado. Primeras filas:")
print(df.head())

Ejercicio Bloque 1: "Cambiando la escala del negocio"
Objetivo: Entender cómo la semilla y el tamaño de la muestra afectan los datos.

Cambia el valor de n (cantidad de datos) a 500 y modifica la "semilla" 42 por tu número de la suerte.

Pregunta técnica: ¿Los montos siguen siendo los mismos? ¿Por qué es importante fijar la semilla en un equipo de trabajo?








---



# Bloque 2: Limpieza e Imputación (El "Pipeline" manual)
Temas: 2, 3, 10, 11, 12

1. Limpieza de Strings (Normalización)
Python
df['cliente'] = df['cliente'].str.strip().str.title()
¿Qué hace?: El accesor .str permite aplicar funciones de texto a toda la columna a la vez (vectorización).

strip(): Elimina espacios "invisibles" al principio y al final (ejemplo: "  Lucho " → "Lucho"). Es vital para que luego los filtros funcionen bien.

title(): Pone la primera letra en mayúscula y el resto en minúscula. Esto estandariza los nombres que pudieron entrar como "LUCHO" o "lucho".

2. Detección y Tratamiento de Nulos (Imputación)
Cuando faltan datos (NaN), no podemos simplemente ignorarlos porque romperían nuestros cálculos.

Mediana en monto: Usamos la mediana (median()) en lugar del promedio porque la mediana no se deja afectar por valores extremadamente altos o bajos. Es más "justa" para representar el centro de los datos.

Moda en categoria: Como la categoría es texto, no podemos sacar un promedio. Usamos la moda (mode()[0]), que es simplemente el valor que más veces aparece (el más frecuente).

3. Transformaciones Vectorizadas (np.where)
Python
df['segmento_venta'] = np.where(df['monto'] > 300, 'Alta', 'Normal')
¿Qué hace?: Funciona exactamente como un SI de Excel.

La ventaja: En lugar de recorrer fila por fila con un bucle (que es lento), np.where procesa miles de filas en milisegundos. Es la forma profesional de crear nuevas categorías basadas en condiciones.

4. Filtrado Avanzado con .loc
Python
df_moda = df.loc[(df['categoria'] == 'Moda') & (df['monto'] > 100)]
loc: Es el estándar de oro en Pandas para seleccionar datos. Significa "localizar".

Las Máscaras Booleanas: Cada condición entre paréntesis genera una lista de "Verdaderos" y "Falsos".

El operador & (AND): Obliga a que se cumplan ambas condiciones a la vez. Es muy potente para segmentar clientes o productos específicos (en este caso, solo artículos de moda caros).



In [60]:
# 1. Limpieza de strings (Tema 9)
df['cliente'] = df['cliente'].str.strip().str.title()

# 2. Detección y tratamiento de nulos (Temas 2, 3)
# Imputación simple: Llenamos monto con la mediana
mediana_monto = df['monto'].median()
df['monto'] = df['monto'].fillna(mediana_monto)

# Imputación por moda para categorías (Tema 3)
df['categoria'] = df['categoria'].fillna(df['categoria'].mode()[0])

# 3. Transformaciones vectorizadas y lógica condicional (Tema 12)
# Clasificamos la venta usando np.where
df['segmento_venta'] = np.where(df['monto'] > 300, 'Alta', 'Normal')

# 4. Filtrado avanzado con .loc (Tema 10)
# Solo ventas de 'Moda' mayores a 100
df_moda = df.loc[(df['categoria'] == 'Moda') & (df['monto'] > 100)]

print(f"Nulos tras limpieza: {df.isnull().sum().sum()}")

Nulos tras limpieza: 0


Ejercicio Bloque 2: "Criterio de Segmentación"
Objetivo: Manipular la lógica condicional y la limpieza de texto.

 1.  En la limpieza de strings, usa .str.upper() en lugar de .str.title() para que los nombres queden todos en MAYÚSCULAS.
2.  Modifica el np.where: ahora las ventas se consideran "VIP" si el monto es mayor a 400, y "Estándar" si es menor.

Pregunta técnica: Al ejecutar el value_counts(), ¿cuántos clientes VIP tenemos ahora en comparación con el código original?



---



In [61]:
# 1. Nombres en MAYÚSCULAS
df['cliente'] = df['cliente'].str.strip().str.upper()

# 2. Nueva lógica VIP (> 400)
df['segmento_venta'] = np.where(df['monto'] > 400, 'VIP', 'Estándar')

# Verificación de resultados
print(df['segmento_venta'].value_counts())

segmento_venta
Estándar    93
VIP          7
Name: count, dtype: int64


# Bloque 3: Análisis Temporal (Series de Tiempo)
Temas: 5, 6, 7

.set_index('fecha'): Para que Pandas entienda que estamos analizando el tiempo, la fecha no puede ser una columna común; debe ser el "eje" o índice del DataFrame.

.resample('W').sum(): "Resamplear" es como alejar la cámara. Pasamos de ver fotos diarias a ver el resumen semanal. Es una agregación temporal.
Código Frecuencia Descripción D Diario Agrupa por día calendario.
W SemanalAgrupa por semanas (termina en domingo).ME MensualAgrupa por fin de mes (Month End).Q TrimestralAgrupa por trimestres (Quarter).YE AnualAgrupa por fin de año (Year End).

Personalización (Poder avanzado)Lo más interesante de Pandas es que podés combinar números con estas letras.
Por ejemplo:resample('2W'): Agruparía los datos cada dos semanas (quincenal).

resample('W-MON'):
Agruparía por semanas, pero haciendo que la semana termine los lunes en lugar de los domingos.


.rolling(window=7).mean(): A diferencia del resample, aquí no alejamos la cámara, sino que usamos un filtro de suavizado. Para cada día, miramos los 6 anteriores y promediamos. Sirve para eliminar el "ruido" de un día malo y ver la tendencia real.

.shift(1): Mueve todos los datos un lugar hacia abajo. Al restar el valor de hoy menos el de ayer (shift(1)), obtenemos la variación diaria.


In [ ]:
# 1. Preparar el índice temporal (Tema 5)
# Usamos un condicional para que solo intente setear el índice si la columna existe
if 'fecha' in df.columns:
    df['fecha'] = pd.to_datetime(df['fecha']) # Aseguramos que sea datetime
    df = df.set_index('fecha')
    print("Índice temporal establecido correctamente.")
else:
    print("El índice ya es temporal o la columna 'fecha' no existe.")

# 2. Resampling y Rolling (Tema 7)
# Suma de ventas semanal
ventas_semanales = df['monto'].resample('W').sum()

# Media móvil de 7 días
df['monto_suavizado'] = df['monto'].rolling(window=7).mean()

# 3. Shift (Variación respecto al día anterior) (Tema 6)
df['dif_dia_anterior'] = df['monto'] - df['monto'].shift(1)

print("Análisis temporal completado.")
print(df[['monto', 'monto_suavizado']].head(10))

# Gráfico rápido de comparación
plt.figure(figsize=(12, 5))
plt.plot(df.index, df['monto'], label='Venta Diaria (Ruido)', alpha=0.4, color='gray')
plt.plot(df.index, df['monto_suavizado'], label='Tendencia (Media Móvil 7d)', color='blue', linewidth=2)
plt.title('Ventas Diarias vs. Tendencia Suavizada')
plt.legend()

# Comparar ventas diarias vs semanales
ventas_semanales = df['monto'].resample('W').sum()

plt.figure(figsize=(10, 4))
ventas_semanales.plot(kind='bar', color='orange', alpha=0.7)
plt.title('Ventas Totales por Semana (Resampling)')
plt.ylabel('Monto Total')
plt.show()

Ejercicio Bloque 3: "Cambiando la mirada temporal"
Objetivo: Entender la diferencia entre frecuencias y ventanas.

1.  Cambia el resample('W') por resample('ME').
2.  Cambia el window=7 del rolling por window=15.

Pregunta técnica: ¿La línea de tendencia azul se volvió más suave o más "nerviosa" al subir el window a 15? ¿Qué pasó con la cantidad de barras en el gráfico de ventas?

In [ ]:
# 1. Cambio de frecuencia a Mensual (Month End)
ventas_mensuales = df['monto'].resample('ME').sum()

# 2. Suavizado más agresivo (Ventana de 15 días)
df['monto_suavizado_15d'] = df['monto'].rolling(window=15).mean()

# Gráfico para ver el impacto
plt.figure(figsize=(10, 4))
plt.plot(df.index, df['monto'], alpha=0.3, label='Original')
plt.plot(df.index, df['monto_suavizado_15d'], color='red', label='Rolling 15d')
plt.title('Impacto de Ventana Larga (15 días)')
plt.legend()
plt.show()



---


# Bloque 4: Visualización y Storytelling
Temas: 13, 14, 15, 16

plt.hist: Nos muestra la forma de los datos. ¿La mayoría de nuestras ventas son
pequeñas o grandes? El histograma nos da la respuesta visual rápida.

px.line (Plotly): Crea gráficos interactivos. La interactividad es parte del storytelling porque permite que el usuario explore los datos (hacer zoom, ver valores exactos al pasar el mouse) en lugar de solo mirar una imagen estática.

add_annotation: Aquí es donde dejas de ser un programador y te vuelves un consultor. Marcar un punto específico con texto ayuda a comunicar el hallazgo estratégico (por ejemplo: "aquí lanzamos la promoción").

In [ ]:
# 1. Matplotlib: Análisis de Distribución (Tema 13)
plt.figure(figsize=(20, 4))
plt.hist(df['monto'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribución de los Montos de Venta')
plt.xlabel('Monto (USD)')
plt.show()

# 2. Plotly: Storytelling Interactivo (Tema 14, 15)
# Creamos un gráfico que explique la evolución
fig = px.line(df.reset_index(), x='fecha', y='monto_suavizado',
              title='Tendencia de Ventas 2026: Crecimiento Estable',
              labels={'monto_suavizado': 'Ventas (Media Móvil)'},
              template='plotly_white')

# Añadimos un mensaje estratégico (Tema 16)
fig.add_annotation(x='2026-04-01', y=df['monto_suavizado'].max(),
            text="Punto máximo de ventas del trimestre",
            showarrow=True, arrowhead=1)

fig.show()

In [ ]:
fig = px.line(df.reset_index(), x='fecha', y='monto_suavizado',
              title='Evolución Mensual de Ventas (Tendencia)')

# Configuramos el eje X para que muestre etiquetas cada 1 mes
fig.update_xaxes(
    dtick="M1",           # "M1" significa un intervalo de 1 Mes
    tickformat="%b %Y",   # %b es el nombre corto del mes (Ene, Feb...)
    ticklabelmode="period"
)

fig.show()

Ejercicio Bloque 4: "Personalización de Storytelling"
Objetivo: Usar la interactividad para resaltar hallazgos.

1.  Cambia el color del gráfico de Plotly: en lugar de 'plotly_white', usa el template 'plotly_dark'.
2.  Mueve la anotación: busca un nuevo punto en la fecha (por ejemplo, '2023-02-15') y cambia el texto para avisar que hubo un "Evento de Marketing".

Pregunta técnica: ¿Cómo cambia la percepción del dato al usar un fondo oscuro? ¿Es más legible para una presentación ejecutiva?

In [ ]:
# 1. Aseguramos que la fecha sea el índice para poder buscarla
df_temp = df.copy()

# 2. Buscamos una fecha que sepamos que existe (la mediana del índice o una posición fija)
# En lugar de escribir '2023-02-15', usamos una posición para que nunca falle:
posicion = len(df_temp) // 2  # Buscamos el punto medio del dataset
fecha_existente = df_temp.index[posicion]
valor_y = df_temp['monto_suavizado'].iloc[posicion]

# 3. Gráfico con Storytelling
fig = px.line(df_temp.reset_index(), x='fecha', y='monto_suavizado',
              title='Reporte de Tendencia - Modo Oscuro',
              template='plotly_dark')

fig.add_annotation(x=fecha_existente,
                   y=valor_y,
                   text="Evento de Marketing",
                   showarrow=True,
                   arrowhead=1,
                   arrowcolor="yellow")

fig.show()

🚩 El Ejercicio: "El Semáforo de Ventas"
Objetivo: Limpiar una pequeña lista de ventas y avisar visualmente cuáles son bajas y cuáles son altas.


1.  Limpieza: Quitar los espacios vacíos en los nombres de los clientes.

1. Completar datos: Si falta un precio (monto), poner el valor promedio de la lista.

2.   Clasificar: Crear una columna "Estado" que diga "Festejar" si la venta es mayor a 300 y "Mejorar" si es menor.

2.   Visualizar: Hacer un gráfico de barras donde el color cambie según ese "Estado".














In [ ]:
# 1. Limpieza básica
df['cliente'] = df['cliente'].str.strip()

# 2. Rellenar nulos con el promedio (lo más simple)
promedio = df['monto'].mean()
df['monto'] = df['monto'].fillna(promedio)

# 3. El "Semáforo" (Lógica condicional)
df['estado'] = np.where(df['monto'] > 300, 'Festejar', 'Mejorar')

# 4. Gráfico de barras interactivo
import plotly.express as px

fig = px.bar(df,
             x='id_transaccion',
             y='monto',
             color='estado',
             title='Mi Primer Semáforo de Ventas',
             color_discrete_map={'Festejar': 'green', 'Mejorar': 'red'})

fig.show()



---


Bloque 2.5 (Nuevo): Automatización con Scikit-Learn (para ver cómo se hace a escala profesional).

Pipeline: Imaginalo como una cinta transportadora de una fábrica. El dato entra sucio por un lado, pasa por varias máquinas (imputador, escalador) y sale limpio por el otro. Evita errores humanos y asegura que el proceso sea siempre el mismo.

SimpleImputer: Es la herramienta que automatiza el llenado de nulos. Al usarlo dentro de un objeto, podés "entrenarlo" con tus datos actuales y usarlo después para datos nuevos que lleguen mañana.

ColumnTransformer: Es el "director de orquesta". Le dice al código: "A las columnas de dinero tratalas con la mediana, pero a las columnas de texto tratalas con la moda". Permite aplicar diferentes estrategias en un solo paso.

In [ ]:
# ==========================================
# BLOQUE 2.5: Imputación Avanzada (Tema 4)
# ==========================================
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Definimos las columnas por tipo
# Usamos el DF original o una copia para no arrastrar la limpieza manual
df_ml = df.copy().reset_index()

num_cols = ['monto']
cat_cols = ['categoria']

# 2. Construimos los Pipelines de tratamiento
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# 3. El Transformador de Columnas (El "Cerebro" del proceso)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ],
    remainder='passthrough' # Mantiene las columnas que no transformamos (como 'fecha')
)

# 4. Ejecutamos la "cañería"
# fit_transform aprende la mediana/moda y la aplica
datos_array = preprocessor.fit_transform(df_ml)

# Convertimos de vuelta a DataFrame para que los alumnos lo entiendan mejor
# (Scikit-learn suele devolver arrays de NumPy)
df_final = pd.DataFrame(datos_array, columns=num_cols + cat_cols + ['id_transaccion', 'fecha', 'cliente', 'segmento_venta'])

print("Pipeline ejecutado. Datos imputados de forma profesional.")
print(df_final.head())